In [2]:
import pandas as pd # import pandas library for data manipulation and analysis
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

# Use Path.cwd().parent to go up 1 folder level
BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

#print(BASE_DIR)
Telco_cust_churn_data = pd.read_csv(DATA_PATH) # read the CSV file into a pandas DataFrame


In [3]:
# Drop identifier columns that are not useful for model training
columns_to_drop = ['customerID']
Telco_cust_churn_data = Telco_cust_churn_data.drop(columns=columns_to_drop, errors='ignore')

print('Columns after dropping:')
print(Telco_cust_churn_data.columns.tolist())

Columns after dropping:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [4]:
# Feature engineering
# 1) Create a tenure group to capture customer loyalty
Telco_cust_churn_data['tenure_group'] = pd.cut(
    Telco_cust_churn_data['tenure'],
    bins=[0, 12, 24, 48, float('inf')],
    labels=['0-12', '13-24', '25-48', '48+'],
    right=False
)

# 2) Create a ratio showing monthly charges relative to customer tenure
Telco_cust_churn_data['monthly_charge_per_tenure'] = (
    Telco_cust_churn_data['MonthlyCharges'] / Telco_cust_churn_data['tenure'].replace(0, 1)
)

print('Engineered columns added:', ['tenure_group', 'monthly_charge_per_tenure'])

# Separate target and feature columns
X = Telco_cust_churn_data.drop(columns=['Churn'], errors='ignore')
y = Telco_cust_churn_data['Churn'] if 'Churn' in Telco_cust_churn_data.columns else None

# Separate numerical and categorical columns
numeric_columns = X.select_dtypes(include=['number']).columns.tolist()
categorical_columns = X.select_dtypes(exclude=['number']).columns.tolist()

print('Numerical columns:', numeric_columns)
print('Categorical columns:', categorical_columns)

# Encode target variable using label encoding
if y is not None:
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    print('Target labels:', label_encoder.classes_)
    print('Encoded target sample:', y_encoded[:10])
else:
    y_encoded = None
    print('No target column named Churn found.')

# One-hot encode categorical features while keeping numerical columns unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_columns)
    ]
)

X_preprocessed = preprocessor.fit_transform(X)
print('Shape after preprocessing:', X_preprocessed.shape)

# Optional: convert back to DataFrame for easier inspection
# encoded_df = pd.DataFrame(X_preprocessed, columns=preprocessor.get_feature_names_out())
# print(encoded_df.head())

Engineered columns added: ['tenure_group', 'monthly_charge_per_tenure']
Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'monthly_charge_per_tenure']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'tenure_group']
Target labels: ['No' 'Yes']
Encoded target sample: [0 0 1 0 1 1 0 0 1 0]
Shape after preprocessing: (7043, 6580)


In [5]:


# Use stratify to preserve the churn class ratio in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_preprocessed,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print('Training set shape:', X_train.shape)
print('Test set shape:', X_test.shape)
print('Train class distribution:', pd.Series(y_train).value_counts().sort_index().to_dict())
print('Test class distribution:', pd.Series(y_test).value_counts().sort_index().to_dict())

Training set shape: (5634, 6580)
Test set shape: (1409, 6580)
Train class distribution: {0: 4139, 1: 1495}
Test class distribution: {0: 1035, 1: 374}
